In [2]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import pymysql
import pandas as pd

load_dotenv()  # .env 파일 읽어서 환경변수로 등록

conn = pymysql.connect(
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    database=os.getenv("DB_NAME"),
    charset="utf8mb4"
)




query_main = """
SELECT
    t.sido, t.sigungu,
    COUNT(*) AS toilet_count,
    SUM(t.total_seats) AS total_seats,
    p.total_pop
FROM tb_toilet t
JOIN tb_population p ON t.sido = p.sido AND t.sigungu = p.sigungu
GROUP BY t.sido, t.sigungu, p.total_pop
"""

df = pd.read_sql(query_main, conn)
print(df.shape)
df.head()

C:\Users\CY\AppData\Local\Temp\ipykernel_19584\1940760838.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_main, conn)


(251, 5)


,sido,sigungu,toilet_count,total_seats,total_pop
0,강원특별자치도,강릉시,379,4976.0,205666
1,강원특별자치도,고성군,241,2023.0,27204
2,강원특별자치도,동해시,204,2328.0,85589
3,강원특별자치도,삼척시,176,1927.0,60294
4,강원특별자치도,속초시,233,2577.0,78893


In [3]:
# ===================================================
# 1. 정규화 지표: 인구 1만 명당 변기 수 (절대량 대신 비율로 비교)
# ===================================================
df['seats_per_10k'] = df['total_seats'] * 10000 / df['total_pop']

In [4]:
# ===================================================
# 2. z-score 표준화: 평균 대비 몇 표준편차 떨어져 있는지
#    (지역 간 상대적 위치를 표준 단위로 비교하기 위함)
# ===================================================
df['seats_z'] = (df['seats_per_10k'] - df['seats_per_10k'].mean()) / df['seats_per_10k'].std()

In [5]:
# ===================================================
# 3. np.where: z-score 기준으로 이상치/평범/취약 지역 분류
#    (극단적으로 밀도 높은 지역 vs 낮은 지역을 빠르게 식별)
# ===================================================
df['tier'] = np.where(
    df['seats_z'] >= 1, '상위 이상치',
    np.where(df['seats_z'] <= -1, '하위 취약지역', '평균 범위')
)

In [6]:
# ===================================================
# 4. np.log1p: 인구(total_pop)가 한쪽으로 치우친(skewed) 분포이므로
#    로그 변환해서 분포를 완만하게 만듦 (시각화·상관분석 왜곡 방지)
# ===================================================
df['log_total_pop'] = np.log1p(df['total_pop'])

In [7]:

# ===================================================
# 5. np.percentile: 밀도 지표의 사분위수 확인 (극단값 파악)
# ===================================================
p25, p50, p75 = np.percentile(df['seats_per_10k'], [25, 50, 75])
print(f"하위 25%: {p25:.2f} / 중앙값: {p50:.2f} / 상위 25%: {p75:.2f}")

하위 25%: 79.72 / 중앙값: 139.92 / 상위 25%: 285.03


In [8]:

# ===================================================
# 6. 상관계수: 인구 규모와 밀도 지표 사이에 관계가 있는지 확인
#    (있다면 "인구 많은 곳이 밀도도 높다"는 뻔한 패턴,
#     상관이 약하면 인구와 무관하게 형평성 문제가 있다는 뜻)
# ===================================================
corr = df['log_total_pop'].corr(df['seats_per_10k'])
print(f"log(인구) vs 밀도 상관계수: {corr:.3f}")

print(df[['sido','sigungu','seats_per_10k','seats_z','tier']]
      .sort_values('seats_z', ascending=False)
      .head(10))

log(인구) vs 밀도 상관계수: -0.698
        sido sigungu  seats_per_10k   seats_z    tier
238     충청북도     단양군    1022.232237  3.939697  상위 이상치
112    대구광역시     군위군     977.809212  3.722261  상위 이상치
15   강원특별자치도     화천군     900.394172  3.343338  상위 이상치
223     충청남도     금산군     884.161916  3.263887  상위 이상치
12   강원특별자치도     태백시     880.171688  3.244356  상위 이상치
13   강원특별자치도     평창군     829.636314  2.997001  상위 이상치
10   강원특별자치도     정선군     828.398998  2.990945  상위 이상치
1    강원특별자치도     고성군     743.640641  2.576080  상위 이상치
95      경상북도     영양군     743.175557  2.573803  상위 이상치
180    인천광역시     옹진군     741.468459  2.565448  상위 이상치
